# Implement Selective State Space Model (Mamba Block) - SOLUTION

**Difficulty**: 🔴 Hard

**Companies**: DeepMind, Google, Anthropic, AI Startups

---

### Problem Statement

State Space Models (SSMs) provide an alternative to Transformers for sequence modeling, offering linear-time complexity in sequence length. **Mamba** (S6) introduces **selective** state spaces where the SSM parameters B and C are input-dependent, allowing the model to selectively focus on or ignore parts of the input.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --- Data: Selective Copying Task ---

torch.manual_seed(42)

def generate_selective_copy_data(n_samples, seq_len=20, vocab_size=8, n_copies=3):
    """
    Generate selective copying task data.

    Tokens 0..vocab_size-1 are regular tokens.
    Token `vocab_size` is the COPY marker.
    Token `vocab_size+1` is PAD/blank.
    """
    blank = vocab_size + 1
    marker = vocab_size

    inputs = torch.full((n_samples, seq_len), blank, dtype=torch.long)
    targets = torch.full((n_samples, seq_len), blank, dtype=torch.long)

    for i in range(n_samples):
        content_len = seq_len - n_copies
        tokens = torch.randint(0, vocab_size, (content_len,))
        inputs[i, :content_len] = tokens

        mark_positions = torch.randperm(content_len)[:n_copies].sort().values

        marked_tokens = []
        for pos in mark_positions:
            inputs[i, pos] = marker
            if pos + 1 < content_len:
                marked_tokens.append(tokens[pos + 1] if pos + 1 < len(tokens) else tokens[pos])
            else:
                marked_tokens.append(tokens[pos])

        for j, tok in enumerate(marked_tokens):
            targets[i, seq_len - n_copies + j] = tok

    return inputs, targets


vocab_size = 8
seq_len = 20
n_copies = 3
train_inputs, train_targets = generate_selective_copy_data(2000, seq_len, vocab_size, n_copies)
test_inputs, test_targets = generate_selective_copy_data(500, seq_len, vocab_size, n_copies)

print(f'Input shape: {train_inputs.shape}')
print(f'Example input:  {train_inputs[0].tolist()}')
print(f'Example target: {train_targets[0].tolist()}')
print(f'Vocab: 0-{vocab_size-1}=tokens, {vocab_size}=MARK, {vocab_size+1}=BLANK')

In [ ]:
# --- Part 1: Selective Scan ---

def selective_scan(x, delta, A, B, C, D):
    """
    Selective scan (S6) - the core SSM recurrence with input-dependent parameters.

    Args:
        x (Tensor): Input, shape (B, L, D).
        delta (Tensor): Step size, shape (B, L, D). Input-dependent.
        A (Tensor): State matrix, shape (D, N). Learned parameter.
        B (Tensor): Input matrix, shape (B, L, N). Input-dependent.
        C (Tensor): Output matrix, shape (B, L, N). Input-dependent.
        D (Tensor): Skip connection, shape (D,). Learned parameter.

    Returns:
        y (Tensor): Output, shape (B, L, D).
    """
    batch, seq_len, d_inner = x.shape
    n = A.shape[1]  # state dimension

    # Step 1: Discretize A and B using ZOH
    # delta: (B, L, D) -> (B, L, D, 1), A: (D, N) -> (1, 1, D, N)
    delta_A = delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0)  # (B, L, D, N)
    A_bar = torch.exp(delta_A)  # (B, L, D, N)

    # B: (B, L, N) -> (B, L, 1, N), delta: (B, L, D) -> (B, L, D, 1)
    B_bar = delta.unsqueeze(-1) * B.unsqueeze(2)  # (B, L, D, N)

    # Step 2: Run the recurrence
    h = torch.zeros(batch, d_inner, n, device=x.device)  # (B, D, N)
    ys = []

    for t in range(seq_len):
        # h = A_bar * h + B_bar * x
        h = A_bar[:, t] * h + B_bar[:, t] * x[:, t, :, None]  # (B, D, N)
        # y = C * h (sum over state dim)
        y_t = (h * C[:, t, None, :]).sum(-1)  # (B, D)
        ys.append(y_t)

    y = torch.stack(ys, dim=1)  # (B, L, D)

    # Step 3: Add skip connection
    y = y + D.unsqueeze(0).unsqueeze(0) * x

    return y


print('selective_scan defined.')

In [ ]:
# --- Part 2: Mamba Block ---

class MambaBlock(nn.Module):
    """
    Mamba block: the core building block of the Mamba architecture.

    Architecture:
        input -> linear proj (expand) -> split into two paths
        Path 1: Conv1d -> SiLU -> SSM (selective scan)
        Path 2: SiLU (gate)
        Combine: path1 * path2 -> linear proj (contract) -> output
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.d_inner = d_model * expand

        # Input projection: d_model -> 2 * d_inner (for x and gate paths)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)

        # Conv1d on the x path (depthwise separable)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, d_conv,
            padding=d_conv - 1, groups=self.d_inner
        )

        # SSM parameter projections from x (input-dependent delta, B, C)
        # Projects to: 1 (delta) + d_state (B) + d_state (C)
        self.x_proj = nn.Linear(self.d_inner, 1 + d_state * 2, bias=False)

        # Project scalar delta to d_inner dimensions
        self.delta_proj = nn.Linear(1, self.d_inner)

        # Learnable SSM parameters
        # A: initialized as negative log-spaced values
        A = torch.arange(1, d_state + 1).float().repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))

        # D: skip connection
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        # Layer norm
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        Args:
            x (Tensor): Input, shape (B, L, d_model).

        Returns:
            Tensor: Output, shape (B, L, d_model).
        """
        x = self.norm(x)
        batch, seq_len, _ = x.shape

        # Project and split into two paths
        xz = self.in_proj(x)  # (B, L, 2*d_inner)
        x_path, z = xz.chunk(2, dim=-1)  # each (B, L, d_inner)

        # Conv1d on x path (causal: truncate future)
        x_conv = self.conv1d(x_path.transpose(1, 2))[:, :, :seq_len].transpose(1, 2)
        x_conv = F.silu(x_conv)

        # Project to get input-dependent delta, B, C
        ssm_params = self.x_proj(x_conv)  # (B, L, 1 + 2*d_state)
        delta_raw = ssm_params[:, :, :1]  # (B, L, 1)
        B = ssm_params[:, :, 1:1+self.d_state]  # (B, L, d_state)
        C = ssm_params[:, :, 1+self.d_state:]  # (B, L, d_state)

        # Apply softplus to delta, then project to d_inner
        delta = F.softplus(self.delta_proj(delta_raw))  # (B, L, d_inner)

        # Get A (negative exponential of learned log values)
        A = -torch.exp(self.A_log)  # (d_inner, d_state)

        # Run selective scan
        y = selective_scan(x_conv, delta, A, B, C, self.D)

        # Gate with z path
        y = y * F.silu(z)

        # Output projection
        return self.out_proj(y)


# Test
block = MambaBlock(d_model=32, d_state=16, d_conv=4, expand=2)
test_x = torch.randn(2, 10, 32)
test_out = block(test_x)
print(f'MambaBlock: input {test_x.shape} -> output {test_out.shape}')
print(f'Parameters: {sum(p.numel() for p in block.parameters()):,}')

In [ ]:
# --- Part 3: Full Mamba Model and RNN Baseline ---

class MambaModel(nn.Module):
    """Stack of Mamba blocks for sequence modeling."""
    def __init__(self, vocab_size, d_model=32, d_state=16, n_layers=2, output_size=None):
        super().__init__()
        if output_size is None:
            output_size = vocab_size
        self.embedding = nn.Embedding(vocab_size + 2, d_model)
        self.layers = nn.ModuleList([MambaBlock(d_model, d_state) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, output_size)

    def forward(self, x):
        h = self.embedding(x)
        for layer in self.layers:
            h = h + layer(h)  # residual
        h = self.norm(h)
        return self.head(h)


class RNNBaseline(nn.Module):
    """Simple RNN baseline for comparison."""
    def __init__(self, vocab_size, d_model=32, n_layers=2, output_size=None):
        super().__init__()
        if output_size is None:
            output_size = vocab_size
        self.embedding = nn.Embedding(vocab_size + 2, d_model)
        self.rnn = nn.GRU(d_model, d_model, n_layers, batch_first=True)
        self.head = nn.Linear(d_model, output_size)

    def forward(self, x):
        h = self.embedding(x)
        h, _ = self.rnn(h)
        return self.head(h)


full_vocab = vocab_size + 2
mamba_model = MambaModel(vocab_size, d_model=32, d_state=16, n_layers=2, output_size=full_vocab)
rnn_model = RNNBaseline(vocab_size, d_model=32, n_layers=2, output_size=full_vocab)

print(f'Mamba parameters: {sum(p.numel() for p in mamba_model.parameters()):,}')
print(f'RNN parameters: {sum(p.numel() for p in rnn_model.parameters()):,}')

In [ ]:
# --- Part 4: Training ---

def train_model(model, train_inputs, train_targets, n_epochs=100, batch_size=128, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    model.train()
    for epoch in range(n_epochs):
        perm = torch.randperm(len(train_inputs))
        epoch_loss = 0
        n_batches = 0

        for i in range(0, len(train_inputs), batch_size):
            x = train_inputs[perm[i:i+batch_size]]
            y = train_targets[perm[i:i+batch_size]]

            logits = model(x)
            loss = F.cross_entropy(logits[:, -n_copies:].reshape(-1, logits.size(-1)),
                                  y[:, -n_copies:].reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        if (epoch + 1) % 25 == 0:
            print(f'  Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.4f}')

    return losses


print('Training Mamba model...')
mamba_losses = train_model(mamba_model, train_inputs, train_targets, n_epochs=100)

print('\nTraining RNN baseline...')
rnn_losses = train_model(rnn_model, train_inputs, train_targets, n_epochs=100)

In [ ]:
# --- Validation ---

def evaluate_model(model, test_inputs, test_targets, n_copies):
    model.eval()
    with torch.no_grad():
        logits = model(test_inputs)
        preds = logits[:, -n_copies:].argmax(dim=-1)
        targets = test_targets[:, -n_copies:]

        token_acc = (preds == targets).float().mean().item()
        seq_acc = (preds == targets).all(dim=-1).float().mean().item()

    return token_acc, seq_acc


mamba_token_acc, mamba_seq_acc = evaluate_model(mamba_model, test_inputs, test_targets, n_copies)
rnn_token_acc, rnn_seq_acc = evaluate_model(rnn_model, test_inputs, test_targets, n_copies)

print('=== Validation ===')
print(f'Mamba  - Token Acc: {mamba_token_acc:.4f}, Sequence Acc: {mamba_seq_acc:.4f}')
print(f'RNN    - Token Acc: {rnn_token_acc:.4f}, Sequence Acc: {rnn_seq_acc:.4f}')

plt.figure(figsize=(8, 5))
plt.plot(mamba_losses, label='Mamba')
plt.plot(rnn_losses, label='RNN Baseline')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Selective Copying Task: Training Loss')
plt.legend()
plt.show()

assert mamba_losses[-1] < mamba_losses[0], 'Mamba loss should decrease!'
print('PASSED: Mamba loss decreased during training')

assert mamba_token_acc > 0.3, f'Mamba token accuracy too low: {mamba_token_acc:.4f}'
print(f'PASSED: Mamba achieves reasonable accuracy ({mamba_token_acc:.4f})')

print('\nAll validations passed!')